# 03 — MDN-LSTM probabilistic forecasting

## Goal

Convert the first 60 days of nine-variable observations into an end-cycle harvest-health distribution and Day-60 PHRI.

**Used for:** uncertainty-aware ranking, probability calibration, PHRI, and daily monitoring.  
**Produces:** `forecast_training_results.csv` and `forecast_model.json`.


## Context & Methods

AQUASURE needs a distribution, not one forecast. The notebook combines an LSTM sequence encoder with a two-component Gaussian mixture-density head. With PyTorch installed, it trains the full MDN-LSTM. Otherwise, a deterministic NumPy LSTM-state fallback keeps the archive executable and records that mode.

### Key assumptions

Forecast origin is Day 60 of a 120-day cycle. Only days 1–60 are features. Benchmark metrics are evaluated later on 240,000 calibrated synthetic scenarios, not this development sample.


In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
SEED = 20260830
print(f"AQUASURE project root: {ROOT}")


## Data

### 1. Assemble Day-60 sequences and targets


In [ ]:
pond = pd.read_csv(ARTIFACTS / "pond_timeseries.csv")
biology = pd.read_csv(ARTIFACTS / "biological_cycles.csv")
VARIABLES = json.loads((ARTIFACTS / "copula_parameters.json").read_text())["variables"]
LOOKBACK = 60
sequences, targets, labels, cycle_ids, farm_ids = [], [], [], [], []
outcome_by_cycle = biology.set_index("cycle_id")
for (farm_id, cycle_id), group in pond[pond.day <= LOOKBACK].groupby(["farm_id", "cycle_id"], sort=False):
    ordered = group.sort_values("day")
    if len(ordered) != LOOKBACK: continue
    outcome = outcome_by_cycle.loc[cycle_id]
    sequences.append(ordered[VARIABLES].to_numpy())
    targets.append(outcome.harvest_health); labels.append(outcome.severe_event)
    cycle_ids.append(cycle_id); farm_ids.append(farm_id)
X = np.asarray(sequences, dtype=float); y = np.asarray(targets); event = np.asarray(labels, dtype=int)
order = np.argsort(cycle_ids)
X, y, event = X[order], y[order], event[order]
cycle_ids, farm_ids = np.asarray(cycle_ids)[order], np.asarray(farm_ids)[order]
split = int(0.75 * len(X))
mean = X[:split].mean(axis=(0, 1), keepdims=True); std = X[:split].std(axis=(0, 1), keepdims=True) + 1e-8
X_scaled = (X - mean) / std
print(f"Sequences: {X.shape}; train={split}; holdout={len(X)-split}; variables={len(VARIABLES)}")


## Results

### 2. Declare the full trainable MDN-LSTM


In [ ]:
TORCH_AVAILABLE = False
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    class MDNLSTM(nn.Module):
        def __init__(self, n_features=9, hidden_size=32, mixtures=2):
            super().__init__()
            self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True)
            self.pi = nn.Linear(hidden_size, mixtures)
            self.mu = nn.Linear(hidden_size, mixtures)
            self.log_sigma = nn.Linear(hidden_size, mixtures)
        def forward(self, sequence):
            encoded, _ = self.lstm(sequence); state = encoded[:, -1]
            return torch.softmax(self.pi(state), -1), torch.sigmoid(self.mu(state)), torch.exp(torch.clamp(self.log_sigma(state), -5, 1))
    def mdn_loss(target, pi, mu, sigma):
        target = target[:, None]
        density = torch.exp(-0.5*((target-mu)/sigma)**2) / (sigma*math.sqrt(2*math.pi))
        return -torch.log(torch.sum(pi*density, dim=1) + 1e-12).mean()
except ImportError:
    print("PyTorch unavailable; tested NumPy fallback will run.")
print("MDN-LSTM definition compiled; PyTorch available:", TORCH_AVAILABLE)


### 3. Fit the full network or disclosed fallback


In [ ]:
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    model = MDNLSTM(len(VARIABLES), 32, 2); optimizer = torch.optim.Adam(model.parameters(), lr=0.008)
    x_train = torch.tensor(X_scaled[:split], dtype=torch.float32); y_train = torch.tensor(y[:split], dtype=torch.float32)
    model.train()
    for _ in range(80):
        optimizer.zero_grad(); pi_t, mu_t, sigma_t = model(x_train)
        loss = mdn_loss(y_train, pi_t, mu_t, sigma_t); loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad(): pi_all, mu_all, sigma_all = model(torch.tensor(X_scaled, dtype=torch.float32))
    pi_all, mu_all, sigma_all = pi_all.numpy(), mu_all.numpy(), sigma_all.numpy()
    mode = "pytorch_mdn_lstm"
else:
    rng = np.random.default_rng(SEED + 3); hidden_size = 20
    W = rng.normal(0, 0.18, size=(len(VARIABLES)+hidden_size, 4*hidden_size)); b = np.zeros(4*hidden_size); b[hidden_size:2*hidden_size] = 1
    def encode(sequence):
        hidden = np.zeros(hidden_size); cell = np.zeros(hidden_size)
        for observation in sequence:
            gate = np.r_[observation, hidden] @ W + b
            input_gate = 1/(1+np.exp(-gate[:hidden_size])); forget_gate = 1/(1+np.exp(-gate[hidden_size:2*hidden_size]))
            candidate = np.tanh(gate[2*hidden_size:3*hidden_size]); output_gate = 1/(1+np.exp(-gate[3*hidden_size:]))
            cell = forget_gate*cell + input_gate*candidate; hidden = output_gate*np.tanh(cell)
        return hidden
    state = np.asarray([encode(sequence) for sequence in X_scaled]); design = np.column_stack([state, np.ones(len(state))])
    beta = np.linalg.solve(design[:split].T@design[:split] + 0.03*np.eye(design.shape[1]), design[:split].T@y[:split])
    point = np.clip(design@beta, 0.05, 0.98); residual = y[:split]-point[:split]; cut = np.median(residual)
    groups = [residual <= cut, residual > cut]
    mixture_weight = np.asarray([g.mean() for g in groups]); mixture_mean = np.asarray([residual[g].mean() for g in groups]); mixture_std = np.asarray([max(residual[g].std(), 0.01) for g in groups])
    pi_all = np.tile(mixture_weight, (len(X), 1)); mu_all = np.clip(point[:, None]+mixture_mean, 0, 1); sigma_all = np.tile(mixture_std, (len(X), 1))
    mode = "numpy_lstm_state_mdn_fallback"
forecast_mean = np.sum(pi_all*mu_all, axis=1)
H_STAR = json.loads((ARTIFACTS / "biological_model_parameters.json").read_text())["H_star"]
from scipy.special import ndtr
phri_day60 = np.clip(np.sum(pi_all*ndtr((H_STAR-mu_all)/np.maximum(sigma_all, 1e-5)), axis=1), 1e-6, 1-1e-6)
print("Execution mode:", mode)


### 4. Save auditable model outputs


In [ ]:
result = pd.DataFrame({"farm_id": farm_ids, "cycle_id": cycle_ids, "split": np.where(np.arange(len(X)) < split, "train", "holdout"), "actual_harvest_health": y, "severe_event": event, "forecast_harvest_health": forecast_mean, "phri_day60_raw": phri_day60})
result.to_csv(ARTIFACTS / "forecast_training_results.csv", index=False)
model_card = {"execution_mode": mode, "input_variables": VARIABLES, "input_variable_count": len(VARIABLES), "lookback_days": LOOKBACK, "forecast_target": "end-cycle harvest_health distribution", "phri_definition": "P(harvest_health < H_star | observations through Day 60)", "H_star": H_STAR, "full_training_dependency": "torch>=2.2"}
(ARTIFACTS / "forecast_model.json").write_text(json.dumps(model_card, indent=2))
holdout = result[result.split.eq("holdout")]
rmse = np.sqrt(np.mean((holdout.actual_harvest_health-holdout.forecast_harvest_health)**2))
print(f"Development holdout RMSE: {rmse:.4f}"); print(result.head().round(4).to_string(index=False))


## Checks


In [ ]:
assert len(VARIABLES) == 9
assert result.phri_day60_raw.between(0, 1).all()
assert result[["farm_id", "cycle_id"]].duplicated().sum() == 0
assert result.split.eq("holdout").sum() > 100
print("PASS — nine inputs, bounded PHRI, unique cycle grain, and holdout present.")


## Takeaways

This notebook exists because insurance needs uncertainty distributions. Its model card discloses the execution mode. Run notebook 04 next.
